**Figure 2: Tail Latency and Real-Agent Behavior.** (a) reports per-call p50/p95 on
the deterministic workload for the two AgentTX modes; (b) reports end-to-end wall
latency of the real DeepSeek refactor (3 repeats). Results suggest that read tracing
is a tail phenomenon (~2x on p95, minor on p50) and that model latency dominates the
deployed experience: the deterministic workload is the honest worst case.


In [ ]:
# ipython -c "%run plot_tail.ipynb"
import json
# Shared USENIX plotting convention (FAST/OSDI camera-ready).
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import style
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, in cm
SINGLE_COL_WIDTH = STANDARD_WIDTH / 2
DOUBLE_COL_WIDTH = STANDARD_WIDTH

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
style.use('bmh')
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['hatch.linewidth'] = 0.5
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['font.family'] = 'Nimbus Roman'
pd.options.display.max_columns = None

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

with open(RESULTS / 'robustness.json', 'r', encoding='utf-8') as handle:
    runtime_rows = json.load(handle)
with open(RESULTS / 'real_agent_robustness.json', 'r', encoding='utf-8') as handle:
    real_agent = json.load(handle)
runtime = pd.DataFrame([row for row in runtime_rows if row.get('suite') == 'p50_p95'])
runtime = runtime.set_index('mode').loc[['agenttx_without_read_tracing', 'agenttx_full']].reset_index()

fig = plt.figure(dpi=300, figsize=(cm_to_inch(DOUBLE_COL_WIDTH), cm_to_inch(3.0)))
plt.rcParams['axes.grid.axis'] = 'y'
colors = ['#2b2d42', '#8d99ae', '#4ecdc4', '#1a535c', '#ef233c']

# (a) Deterministic workload tail: median versus 95th percentile.
ax0 = plt.subplot(1, 2, 1)
x = np.arange(len(runtime))
bar_width = 0.34
p50 = runtime['step_p50_ms'].astype(float).to_numpy()
p95 = runtime['step_p95_ms'].astype(float).to_numpy()
bars_p50 = ax0.bar(x - bar_width / 2, p50, width=bar_width, color=colors[1], hatch='///', linewidth=0.5, label='p50')
bars_p95 = ax0.bar(x + bar_width / 2, p95, width=bar_width, color=colors[4], linewidth=0.5, label='p95')
ax0.annotate(f"x{p95[1] / p95[0]:.1f}", (1 + bar_width / 2, p95[1]), textcoords='offset points', xytext=(0, 3), ha='center', fontsize=6, color=colors[4])
ax0.set_xticks(x, labels=['AgentTX\nno-trace', 'AgentTX\nfull'], fontsize=7)
ax0.set_ylabel('Per-call latency (ms)', fontsize=8)
ax0.set_title('(a) Deterministic workload tail', fontsize=8)
ax0.tick_params(bottom=False, top=False, left=False, right=False)
ax0.tick_params(axis='y', labelsize=8)

# (b) Real-agent refactor: wall latency with success annotation.
ax1 = plt.subplot(1, 2, 2)
metrics = ['wall_p50_s', 'wall_p95_s']
values = [float(real_agent[metric]) for metric in metrics]
bars = ax1.bar(np.arange(2), values, width=0.55, color=[colors[1], colors[4]], linewidth=0.5, hatch=['///', ''])
ax1.set_xticks(np.arange(2), labels=['wall p50', 'wall p95'], fontsize=7)
ax1.set_ylabel('Task latency (s)', fontsize=8)
ax1.set_title('(b) Real-agent refactor', fontsize=8)
ax1.tick_params(bottom=False, top=False, left=False, right=False)
ax1.tick_params(axis='y', labelsize=8)
ax1.text(0.5, 0.90, f"success={real_agent['success_rate']:.0%}, leak={real_agent['host_leak_rate']:.0%}", transform=ax1.transAxes, ha='center', fontsize=7)

for ax in fig.axes:
    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(0.5)
fig.legend([bars_p50[0], bars_p95[0]], ['p50', 'p95'], loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False, columnspacing=1.0, handlelength=1.2, fontsize=8)
plt.tight_layout(pad=0.4, rect=[0.045, 0.0, 0.99, 0.88])
plt.savefig(FIGDIR / 'FIG-Motivation-Tail.pdf', bbox_inches='tight', pad_inches=0)
plt.savefig(FIGDIR / 'FIG-Motivation-Tail.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

print(f"read tracing tail: p50 x{p50[1] / p50[0]:.2f}, p95 x{p95[1] / p95[0]:.2f}")
